In [0]:
from pyspark.sql import functions as F
spark.sql("USE workspace.taxi_medallion")

bronze = spark.read.table("bronze_trips")

# Rule: a row is valid only if ALL conditions are true
valid = F.coalesce(
    (F.col("fare_amount") > 0) &
    (F.col("trip_distance") > 0) &
    (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime")) &
    F.col("pickup_zip").isNotNull() & F.col("dropoff_zip").isNotNull(),
    F.lit(False))

quarantine = bronze.filter(~valid)
quarantine.write.format("delta").mode("overwrite").saveAsTable("quarantine_trips")

silver = (bronze.filter(valid)
    .withColumn("trip_id", F.sha2(F.concat_ws("|",
        "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "pickup_zip", "dropoff_zip", "fare_amount", "trip_distance"), 256))
    .dropDuplicates(["trip_id"])
    .withColumn("trip_minutes", (F.unix_timestamp("tpep_dropoff_datetime")
                                 - F.unix_timestamp("tpep_pickup_datetime")) / 60)
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("fare_per_mile", F.round(F.col("fare_amount") / F.col("trip_distance"), 2))
    .withColumn("_processed_at", F.current_timestamp()))

silver.write.format("delta").mode("overwrite").saveAsTable("silver_trips")

n_bronze, n_silver, n_bad = bronze.count(), silver.count(), quarantine.count()
print(f"bronze={n_bronze} silver={n_silver} quarantined={n_bad} ({n_bad/n_bronze:.1%})")